In [5]:
import pyreadstat
import pandas as pd

# ── Paths and configs ─────────────────────────────────────────────────────────
year_configs = {
    'y1': {
        'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_y1_2015-16.csv",
        'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav",
        'la_col': 'LA_2021',
        'scheme': 'y1',
    },
    'y2': {
        'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_y2_2016-17.csv",
        'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8391-spss 2016-2017\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav",
        'la_col': 'LA_2020',
        'scheme': 'y2',
    },
    'y3': {
        'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_y3_2017-18.csv",
        'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8651-spss 2017-2018\spss\spss28\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav",
        'la_col': 'LA_Pre2019',
        'scheme': 'b',
    },
    'y4': {
        'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_y4_2018-19.csv",
        'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8652-spss 2018-2019\spss\spss28\active_lives_survey_nov_18-19_data_year_4_shared_20250103.sav",
        'la_col': 'LA_2019',
        'scheme': 'b',
    },
    'y5': {
        'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_y5_2019-20.csv",
        'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8899-spss 2019-2020\spss\spss28\active_lives_survey_nov_19-20_data_year_5_shared_20250103.sav",
        'la_col': 'LA_2019',
        'scheme': 'b',
    },
    'y6': {
        'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_y6_2020-21.csv",
        'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8993-spss 2020-2021\spss\spss28\active_lives_survey_nov_20-21_data_year_6_shared_20250103.sav",
        'la_col': 'LA_2020',
        'scheme': 'y6',
    },
    'y7': {
        'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_y7_2021-22.csv",
        'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9136-spss 2021-2022\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav",
        'la_col': 'LA_2021',
        'scheme': 'y7',
    },
    'y8': {
        'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_year8.csv",
        'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav",
        'la_col': 'LA_2023',
        'scheme': 'a',
    },
}

# ── Borough lookups (from EDA) ─────────────────────────────────────────────────
# Load scheme A (Y1/Y2/Y6/Y7/Y8) and scheme B (Y3/Y4/Y5) lookups
def build_borough_lookup(sav_path, la_col):
    _, meta = pyreadstat.read_sav(sav_path, metadataonly=True)
    labels = meta.variable_value_labels.get(la_col, {})
    return {
        code: name.split(' ', 1)[1]
        for code, name in labels.items()
        if 'E09' in str(name) and 'E09000001' not in str(name)
    }

print("Building borough lookups...")
borough_lookups = {
    'y1': build_borough_lookup(year_configs['y1']['sav'], 'LA_2021'),
    'y2': build_borough_lookup(year_configs['y2']['sav'], 'LA_2020'),
    'b':  build_borough_lookup(year_configs['y4']['sav'], 'LA_2019'),
    'y6': build_borough_lookup(year_configs['y6']['sav'], 'LA_2020'),
    'y7': build_borough_lookup(year_configs['y7']['sav'], 'LA_2021'),
    'a':  build_borough_lookup(year_configs['y8']['sav'], 'LA_2023'),
}
print("✓ Borough lookups ready")

# ── Process each year ─────────────────────────────────────────────────────────
decoded_dfs = {}

for label, config in year_configs.items():
    print(f"\nProcessing {label}...")

    # Load SPSS metadata for this year
    _, meta = pyreadstat.read_sav(config['sav'], metadataonly=True)
    value_labels = meta.variable_value_labels

    # Load CSV
    df = pd.read_csv(config['csv'], low_memory=False)
    print(f"  Loaded: {df.shape}")

    # ── Decode all columns using this year's metadata ─────────────────────────
    for col in df.columns:
        if col in value_labels:
            df[col] = df[col].map(value_labels[col])

    # ── Re-apply EDA fixes ────────────────────────────────────────────────────

    # Fix 1: NSSEC5 — recode 'Aged <16 or 75+' → 'Students and other / unclassified'
    if 'NSSEC5' in df.columns:
        before = (df['NSSEC5'] == 'Aged <16 or 75+').sum()
        df['NSSEC5'] = df['NSSEC5'].replace(
            'Aged <16 or 75+', 'NS SEC 9: Students and other / unclassified'
        )
        if before > 0:
            print(f"  NSSEC5 recode: {before} rows fixed ✓")

    # Fix 2: DURATION1PL_TEAMSPORT_C05 — -1.0 in Y2 (already decoded so check raw)
    # This was a numeric -1.0 with no label so it won't decode to anything
    # It will remain as NaN after decoding — already handled ✓

    # Fix 3: Borough name from LA_code
    borough_lookup = borough_lookups[config['scheme']]
    df['borough'] = df['LA_code'].map(borough_lookup)

    # Fix 4: MEMS7GR_ALL label standardisation
    # Y1-Y4 use 'Insufficiently Active', Y5-Y8 use 'Fairly Active'
    # Standardise all to 'Fairly Active' for consistency
    if 'MEMS7GR_ALL' in df.columns:
        df['MEMS7GR_ALL'] = df['MEMS7GR_ALL'].replace(
            'Insufficiently Active', 'Fairly Active'
        )

    print(f"  Decoded and fixed: {df.shape}")
    decoded_dfs[label] = df

# ── Combine all years ─────────────────────────────────────────────────────────
print("\nCombining all years...")
df_all_decoded = pd.concat(decoded_dfs.values(), ignore_index=True)

print(f"\n✓ Combined shape:    {df_all_decoded.shape}")
print(f"  Unique boroughs:   {df_all_decoded['borough'].nunique()}")
print(f"  Unmapped boroughs: {df_all_decoded['borough'].isna().sum()}")
print(f"  Survey years:      {sorted(df_all_decoded['survey_year'].unique())}")

# ── Quick sense check ─────────────────────────────────────────────────────────
print("\nSample decoded values:")
check_cols = ['Age9', 'Gend3', 'Eth7', 'MEMS7GR_ALL', 'VolAny', 'borough']
for col in check_cols:
    if col in df_all_decoded.columns:
        print(f"\n{col}:")
        print(df_all_decoded[col].value_counts().head(4))

# ── Save ──────────────────────────────────────────────────────────────────────
df_all_decoded.to_csv('active_lives_london_all_years_decoded.csv', index=False)
print("\n✓ Saved: active_lives_london_all_years_decoded.csv")

Building borough lookups...
✓ Borough lookups ready

Processing y1...
  Loaded: (19620, 292)


C:\Users\trish\AppData\Local\Temp\ipykernel_14672\938044059.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['borough'] = df['LA_code'].map(borough_lookup)


  Decoded and fixed: (19620, 293)

Processing y2...
  Loaded: (19248, 498)
  NSSEC5 recode: 1314 rows fixed ✓
  Decoded and fixed: (19248, 499)

Processing y3...


C:\Users\trish\AppData\Local\Temp\ipykernel_14672\938044059.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['borough'] = df['LA_code'].map(borough_lookup)


  Loaded: (15967, 527)


...

C:\Users\trish\AppData\Local\Temp\ipykernel_14672\938044059.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['borough'] = df['LA_code'].map(borough_lookup)


  Decoded and fixed: (16028, 590)

Processing y7...
  Loaded: (16139, 590)


C:\Users\trish\AppData\Local\Temp\ipykernel_14672\938044059.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['borough'] = df['LA_code'].map(borough_lookup)


  Decoded and fixed: (16139, 591)

Processing y8...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\trish\\OneDrive - University of Bristol\\summer project\\TS\\Data\\active_lives_london_year8.csv'

In [6]:
# Fix Y8 path and rerun just Y8
config_y8 = {
    'csv':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_y8_2022-23.csv",
    'sav':  r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav",
    'la_col': 'LA_2023',
    'scheme': 'a',
}

print("Processing y8...")
_, meta = pyreadstat.read_sav(config_y8['sav'], metadataonly=True)
value_labels = meta.variable_value_labels

df = pd.read_csv(config_y8['csv'], low_memory=False)
print(f"  Loaded: {df.shape}")

for col in df.columns:
    if col in value_labels:
        df[col] = df[col].map(value_labels[col])

if 'NSSEC5' in df.columns:
    df['NSSEC5'] = df['NSSEC5'].replace('Aged <16 or 75+', 'NS SEC 9: Students and other / unclassified')

df['borough'] = df['LA_code'].map(borough_lookups['a'])

if 'MEMS7GR_ALL' in df.columns:
    df['MEMS7GR_ALL'] = df['MEMS7GR_ALL'].replace('Insufficiently Active', 'Fairly Active')

print(f"  Decoded and fixed: {df.shape}")
decoded_dfs['y8'] = df

# ── Now combine all years ─────────────────────────────────────────────────────
print("\nCombining all years...")
df_all_decoded = pd.concat(decoded_dfs.values(), ignore_index=True)

print(f"\n✓ Combined shape:    {df_all_decoded.shape}")
print(f"  Unique boroughs:   {df_all_decoded['borough'].nunique()}")
print(f"  Unmapped boroughs: {df_all_decoded['borough'].isna().sum()}")
print(f"  Survey years:      {sorted(df_all_decoded['survey_year'].unique())}")

# Save
df_all_decoded.to_csv('active_lives_london_all_years_decoded.csv', index=False)
print("\n✓ Saved: active_lives_london_all_years_decoded.csv")

Processing y8...
  Loaded: (16515, 601)


C:\Users\trish\AppData\Local\Temp\ipykernel_14672\2967764574.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['borough'] = df['LA_code'].map(borough_lookups['a'])
C:\Users\trish\AppData\Local\Temp\ipykernel_14672\2967764574.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_decoded = pd.concat(decoded_dfs.values(), ignore_index=True)


  Decoded and fixed: (16515, 602)

Combining all years...

✓ Combined shape:    (135497, 610)
  Unique boroughs:   32
  Unmapped boroughs: 0
  Survey years:      ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23']

✓ Saved: active_lives_london_all_years_decoded.csv


In [8]:
# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 25)
pd.set_option('display.width', None)

# Load and display
df_all_decoded = pd.read_csv('active_lives_london_all_years_decoded.csv', low_memory=False, nrows=25)
print(f"Columns: {df_all_decoded.shape[1]}")
df_all_decoded.head(25)

Columns: 610


,serial,mode,group,Quarter,MonthTwelve,wt_final,xStrata,LA_code,LA_2023,LA_Old,Age5_2,Age9,Disab2_POP,Disab3,disty1_POP,disty10_POP,disty2_POP,disty3_POP,disty4_POP,disty5_POP,disty6_POP,disty7_POP,disty8_POP,disty9_POP,Educ6,Eth2,Eth7,EthAge4,Gend3,GendAge9,NSSEC5,NSSEC8,BMIG,MEMS7_ALL,MEMS7GR_ALL,MEMS7_SPORTCOUNT_A01,MEMS7GR_SPORTCOUNT_A01,MEMS7GR7_SPORTCOUNT_A01,MONTHS_12_WALKLEISURE_B01,DAYS10P_WALKLEISURE_B01,DAYS10P60_WALKLEISURE_B01,DAYS10P60GR_WALKLEISURE_B01,MEMS7_CAPPED_WALKLEISURE_B01,ACTYRA_WALKLEISURE_B01,ACTYRB_WALKLEISURE_B01,ACTYRC_WALKLEISURE_B01,FREQUENCY_WALKLEISURE_B01,DURATION_WALKLEISURE_B01,MEMS7_WALKLEISURE_B01,FREQUENCYGR_WALKLEISURE_B01,DURATIONGR_WALKLEISURE_B01,DURATION1PL_WALKLEISURE_B01,MEMS7GR_WALKLEISURE_B01,DAYS10PGR2_WALKLEISURE_B01,MONTHS_12_WALKTRAV_B02,DAYS10P_WALKTRAV_B02,DAYS10P60_WALKTRAV_B02,DAYS10P60GR_WALKTRAV_B02,MEMS7_CAPPED_WALKTRAV_B02,ACTYRA_WALKTRAV_B02,ACTYRB_WALKTRAV_B02,ACTYRC_WALKTRAV_B02,FREQUENCY_WALKTRAV_B02,DURATION_WALKTRAV_B02,MEMS7_WALKTRAV_B02,FREQUENCYGR_WALKTRAV_B02,DURATIONGR_WALKTRAV_B02,DURATION1PL_WALKTRAV_B02,MEMS7GR_WALKTRAV_B02,DAYS10PGR2_WALKTRAV_B02,MONTHS_12_CYCTRAV_B04,DAYS10P_CYCTRAV_B04,DAYS10P60_CYCTRAV_B04,DAYS10P60GR_CYCTRAV_B04,MEMS7_CAPPED_CYCTRAV_B04,ACTYRA_CYCTRAV_B04,ACTYRB_CYCTRAV_B04,ACTYRC_CYCTRAV_B04,FREQUENCY_CYCTRAV_B04,DURATION_CYCTRAV_B04,MEMS7_CYCTRAV_B04,FREQUENCYGR_CYCTRAV_B04,DURATIONGR_CYCTRAV_B04,DURATION1PL_CYCTRAV_B04,MEMS7GR_CYCTRAV_B04,DAYS10PGR2_CYCTRAV_B04,MONTHS_12_DANCEART_B05,DAYS10P_DANCEART_B05,DAYS10P60_DANCEART_B05,DAYS10P60GR_DANCEART_B05,ACTYRA_DANCEART_B05,ACTYRB_DANCEART_B05,ACTYRC_DANCEART_B05,FREQUENCY_DANCEART_B05,DURATION_DANCEART_B05,MEMS7_DANCEART_B05,FREQUENCYGR_DANCEART_B05,DURATIONGR_DANCEART_B05,DURATION1PL_DANCEART_B05,MEMS7GR_DANCEART_B05,DAYS10PGR2_DANCEART_B05,MONTHS_12_SPORTTRAD_B07,DAYS10P_SPORTTRAD_B07,DAYS10P60_SPORTTRAD_B07,DAYS10P60GR_SPORTTRAD_B07,ACTYRA_SPORTTRAD_B07,ACTYRB_SPORTTRAD_B07,ACTYRC_SPORTTRAD_B07,FREQUENCY_SPORTTRAD_B07,DURATION_SPORTTRAD_B07,MEMS7_SPORTTRAD_B07,FREQUENCYGR_SPORTTRAD_B07,DURATIONGR_SPORTTRAD_B07,DURATION1PL_SPORTTRAD_B07,MEMS7GR_SPORTTRAD_B07,MONTHS_12_WALKALL_C01,DAYS10P_WALKALL_C01,DAYS10P60_WALKALL_C01,DAYS10P60GR_WALKALL_C01,MEMS7_CAPPED_WALKALL_C01,ACTYRA_WALKALL_C01,ACTYRB_WALKALL_C01,ACTYRC_WALKALL_C01,FREQUENCY_WALKALL_C01,DURATION_WALKALL_C01,MEMS7_WALKALL_C01,FREQUENCYGR_WALKALL_C01,DURATIONGR_WALKALL_C01,DURATION1PL_WALKALL_C01,MEMS7GR_WALKALL_C01,MONTHS_12_ACTTRAV_C03,DAYS10P_ACTTRAV_C03,DAYS10P60_ACTTRAV_C03,DAYS10P60GR_ACTTRAV_C03,MEMS7_CAPPED_ACTTRAV_C03,ACTYRA_ACTTRAV_C03,ACTYRB_ACTTRAV_C03,ACTYRC_ACTTRAV_C03,FREQUENCY_ACTTRAV_C03,DURATION_ACTTRAV_C03,MEMS7_ACTTRAV_C03,FREQUENCYGR_ACTTRAV_C03,DURATIONGR_ACTTRAV_C03,DURATION1PL_ACTTRAV_C03,MEMS7GR_ACTTRAV_C03,MONTHS_12_DANCEALL_C04,DAYS10P_DANCEALL_C04,DAYS10P60_DANCEALL_C04,DAYS10P60GR_DANCEALL_C04,ACTYRA_DANCEALL_C04,ACTYRB_DANCEALL_C04,ACTYRC_DANCEALL_C04,FREQUENCY_DANCEALL_C04,DURATION_DANCEALL_C04,MEMS7_DANCEALL_C04,FREQUENCYGR_DANCEALL_C04,DURATIONGR_DANCEALL_C04,DURATION1PL_DANCEALL_C04,MEMS7GR_DANCEALL_C04,MONTHS_12_TEAMSPORT_C05,DAYS10P_TEAMSPORT_C05,DAYS10P60_TEAMSPORT_C05,DAYS10P60GR_TEAMSPORT_C05,ACTYRA_TEAMSPORT_C05,ACTYRB_TEAMSPORT_C05,ACTYRC_TEAMSPORT_C05,FREQUENCY_TEAMSPORT_C05,DURATION_TEAMSPORT_C05,MEMS7_TEAMSPORT_C05,FREQUENCYGR_TEAMSPORT_C05,DURATIONGR_TEAMSPORT_C05,DURATION1PL_TEAMSPORT_C05,MEMS7GR_TEAMSPORT_C05,MINS_SESS_TEAMSPORT_C05,MONTHS_12_RACKETSPORT_C06,DAYS10P_RACKETSPORT_C06,DAYS10P60_RACKETSPORT_C06,DAYS10P60GR_RACKETSPORT_C06,ACTYRA_RACKETSPORT_C06,ACTYRB_RACKETSPORT_C06,ACTYRC_RACKETSPORT_C06,FREQUENCY_RACKETSPORT_C06,DURATION_RACKETSPORT_C06,MEMS7_RACKETSPORT_C06,FREQUENCYGR_RACKETSPORT_C06,DURATIONGR_RACKETSPORT_C06,DURATION1PL_RACKETSPORT_C06,MEMS7GR_RACKETSPORT_C06,MONTHS_12_ADVWATERSPORT_C07,DAYS10P_ADVWATERSPORT_C07,DAYS10P60_ADVWATERSPORT_C07,DAYS10P60GR_ADVWATERSPORT_C07,ACTYRA_ADVWATERSPORT_C07,ACTYRB_ADVWATERSPORT_C07,ACTYRC_ADVWATERSPORT_C07,FREQUENCY_ADV

In [11]:
# Load both datasets
df_numeric = pd.read_csv(r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_all_years.csv", low_memory=False)
df_decoded = pd.read_csv('active_lives_london_all_years_decoded.csv', low_memory=False)

print("=== SHAPE CHECK ===")
print(f"Numeric:  {df_numeric.shape}")
print(f"Decoded:  {df_decoded.shape}")

print("\n=== ROW COUNT BY YEAR ===")
print(pd.DataFrame({
    'Numeric': df_numeric.groupby('survey_year').size(),
    'Decoded': df_decoded.groupby('survey_year').size(),
}))

print("\n=== BOROUGH CHECK ===")
print(f"Numeric unique boroughs:  {df_numeric['borough'].nunique()}")
print(f"Decoded unique boroughs:  {df_decoded['borough'].nunique()}")
print(f"Numeric unmapped:         {df_numeric['borough'].isna().sum()}")
print(f"Decoded unmapped:         {df_decoded['borough'].isna().sum()}")

print("\n=== NaN RATES ON KEY VARIABLES ===")
key_vars = ['Age9', 'Gend3', 'Eth7', 'Disab3', 'NSSEC5',
            'MEMS7GR_ALL', 'VolAny', 'health']
print(f"{'Variable':<20} {'Numeric NaN%':>14} {'Decoded NaN%':>14} {'Match':>8}")
print("-" * 60)
for col in key_vars:
    if col in df_numeric.columns and col in df_decoded.columns:
        n_nan = df_numeric[col].isna().mean() * 100
        d_nan = df_decoded[col].isna().mean() * 100
        match = "✓" if abs(n_nan - d_nan) < 0.1 else "⚠ DIFF"
        print(f"  {col:<18} {n_nan:>13.1f}% {d_nan:>13.1f}% {match:>8}")

print("\n=== NSSEC5 RECODE CHECK ===")
print("Numeric NSSEC5 unique values:", sorted(df_numeric['NSSEC5'].dropna().unique()))
print("Decoded NSSEC5 unique values:", sorted(df_decoded['NSSEC5'].dropna().unique()))

print("\n=== MEMS7GR_ALL CONSISTENCY CHECK ===")
print("Decoded MEMS7GR_ALL unique values:")
print(df_decoded['MEMS7GR_ALL'].value_counts())

print("\n=== SAMPLE DECODED VALUES ===")
check_cols = ['Age9', 'Gend3', 'Eth7', 'MEMS7GR_ALL', 'VolAny', 'health', 'borough']
for col in check_cols:
    if col in df_decoded.columns:
        print(f"\n{col}: {df_decoded[col].value_counts().head(3).to_dict()}")

=== SHAPE CHECK ===
Numeric:  (135497, 610)
Decoded:  (135497, 610)

=== ROW COUNT BY YEAR ===
             Numeric  Decoded
survey_year                  
2015-16        19620    19620
2016-17        19248    19248
2017-18        15967    15967
2018-19        15889    15889
2019-20        16091    16091
2020-21        16028    16028
2021-22        16139    16139
2022-23        16515    16515

=== BOROUGH CHECK ===
Numeric unique boroughs:  32
Decoded unique boroughs:  32
Numeric unmapped:         0
Decoded unmapped:         0

=== NaN RATES ON KEY VARIABLES ===
Variable               Numeric NaN%   Decoded NaN%    Match
------------------------------------------------------------
  Age9                         1.0%           1.0%        ✓
  Gend3                        0.2%           0.2%        ✓
  Eth7                         6.8%           6.8%        ✓
  Disab3                       6.7%           6.7%        ✓
  NSSEC5                       6.4%           6.4%        ✓
  MEMS7GR_A

In [12]:
df_all_decoded['NSSEC5'] = df_all_decoded['NSSEC5'].replace(
    'NS SEC 9: Students and other',
    'NS SEC 9: Students and other / unclassified'
)
print("NSSEC5 unique values after fix:")
print(df_all_decoded['NSSEC5'].value_counts())

NSSEC5 unique values after fix:
NSSEC5
NS SEC 1-2: Higher social groups               77300
NS SEC 3-5: Middle social groups               23178
NS SEC 9: Students and other / unclassified    13902
NS SEC 6-8: Lower social groups                12397
Name: count, dtype: int64


In [13]:
df_all_decoded['VolAny'] = df_all_decoded['VolAny'].replace({
    '0.0': 'No',
    '1.0': 'Yes'
})
print("\nVolAny unique values after fix:")
print(df_all_decoded['VolAny'].value_counts())


VolAny unique values after fix:
VolAny
No     70502
Yes    14786
Name: count, dtype: int64


In [14]:
df_all_decoded.to_csv('active_lives_london_all_years_decoded.csv', index=False)
print("\n✓ Saved with fixes applied")


✓ Saved with fixes applied


In [15]:
# Load full dataset
df_all_decoded = pd.read_csv('active_lives_london_all_years_decoded.csv', low_memory=False)

print("wt_final NaN% by year:")
print(df_all_decoded.groupby('survey_year')['wt_final'].apply(
    lambda x: f"{x.isna().mean()*100:.1f}% ({x.isna().sum()} rows)"
))

print(f"\nTotal wt_final NaN: {df_all_decoded['wt_final'].isna().sum():,}")
print(f"Total rows:         {len(df_all_decoded):,}")

# Check if NaN rows have valid data in other columns
nan_wt = df_all_decoded[df_all_decoded['wt_final'].isna()]
print(f"\nSample of rows with NaN wt_final:")
print(nan_wt[['survey_year', 'borough', 'Age9', 'Gend3', 'MEMS7GR_ALL']].head(10))

wt_final NaN% by year:
survey_year
2015-16    100.0% (19620 rows)
2016-17          0.0% (0 rows)
2017-18    100.0% (15967 rows)
2018-19    100.0% (15889 rows)
2019-20    100.0% (16091 rows)
2020-21          0.0% (0 rows)
2021-22          0.0% (0 rows)
2022-23          0.0% (0 rows)
Name: wt_final, dtype: object

Total wt_final NaN: 67,567
Total rows:         135,497

Sample of rows with NaN wt_final:
  survey_year               borough   Age9   Gend3    MEMS7GR_ALL
0     2015-16  Barking and Dagenham  35-44  Female         Active
1     2015-16  Barking and Dagenham    NaN  Female         Active
2     2015-16  Barking and Dagenham  35-44    Male  Fairly Active
3     2015-16  Barking and Dagenham  55-64    Male       Inactive
4     2015-16  Barking and Dagenham  55-64    Male       Inactive
5     2015-16  Barking and Dagenham  16-24    Male         Active
6     2015-16  Barking and Dagenham  55-64    Male         Active
7     2015-16  Barking and Dagenham  45-54    Male         Active
8 

In [16]:
import pyreadstat

# Check weight column names in the affected years
year_savs = {
    '2015-16': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav",
    '2017-18': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8651-spss 2017-2018\spss\spss28\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav",
    '2018-19': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8652-spss 2018-2019\spss\spss28\active_lives_survey_nov_18-19_data_year_4_shared_20250103.sav",
    '2019-20': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8899-spss 2019-2020\spss\spss28\active_lives_survey_nov_19-20_data_year_5_shared_20250103.sav",
}

for year, path in year_savs.items():
    _, meta = pyreadstat.read_sav(path, metadataonly=True)
    # Search for any weight-related column
    weight_cols = [(v, meta.column_names_to_labels.get(v, '')) 
                   for v in meta.column_names 
                   if 'wt' in v.lower() or 'weight' in v.lower()]
    print(f"\n{year}:")
    for col, label in weight_cols:
        print(f"  {col:20} | {label}")


2015-16:
  wt_final             | All: Online and Postal
  wt_final_online      | All: Online
  wt_final_B           | Phase 2: Online Group 1
  wt_final_C           | Phase 2: Online Group 2
  wt_final_AB          | Phase 2: Postal & Online Group 1
  wt_final_AC          | Phase 2: Postal & Online Group 2
  wt_time              | All: Online and Postal - time series weight
  wt_online_time       | All: Online - time series weight
  DVWeightKG           | Weight in kg
  MONTHS_12_WEIGHTSCLASS_P09 | Done in the last 12 months: Weights-based class (ONLINE ONLY)
  DAYS10P_WEIGHTSCLASS_P09 | Num sessions of moderate activity for at least 10 minutes in last 28 days: Weights-based class (ONLINE ONLY)
  DAYS10P60_WEIGHTSCLASS_P09 | Num sessions of moderate activity for at least 10 minutes in last 28 days with 60 minute limit: Weights-based class (ONLINE ONLY)
  DAYS10P60GR_WEIGHTSCLASS_P09 | Num sessions of moderate activity for at least 10 minutes in last 28 days with 60 min limit (grouped)

In [18]:
import pyreadstat

SAV_Y1 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"

_, meta_y1 = pyreadstat.read_sav(SAV_Y1, metadataonly=True)

# Check if wt_final has value labels defined
wt_labels = meta_y1.variable_value_labels.get('wt_final', {})
print(f"wt_final value labels in Y1: {wt_labels}")

# Also check the numeric dataset to confirm wt_final has real values
df_numeric = pd.read_csv(r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_all_years.csv", low_memory=False)
print(f"\nwt_final in numeric dataset by year:")
print(df_numeric.groupby('survey_year')['wt_final'].apply(
    lambda x: f"NaN: {x.isna().sum()}, Mean: {x.mean():.4f}" if x.notna().sum() > 0 else "ALL NaN"
))

wt_final value labels in Y1: {-99.0: 'Missing, should have been answered', -98.0: 'Not applicable: Survey routing', -97.0: 'Incorrectly multicoded', -96.0: 'Out of range', -95.0: "Cannot give an estimate / Don't know", -94.0: 'Prefer not to say', -93.0: 'Not applicable: Postal response', -92.0: 'Not applicable: Phase 1 case', -91.0: 'Not applicable: Phase 2 case', -90.0: 'Not applicable: Not asked to this group'}

wt_final in numeric dataset by year:
survey_year
2015-16    NaN: 0, Mean: 1.5549
2016-17    NaN: 0, Mean: 1.5778
2017-18    NaN: 0, Mean: 1.7532
2018-19    NaN: 0, Mean: 1.7849
2019-20    NaN: 0, Mean: 1.7256
2020-21    NaN: 0, Mean: 1.7294
2021-22    NaN: 0, Mean: 1.7189
2022-23    NaN: 0, Mean: 1.6137
Name: wt_final, dtype: object


In [21]:
# Reload numeric dataset
df_numeric = pd.read_csv(r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_all_years.csv", low_memory=False)
df_decoded = pd.read_csv('active_lives_london_all_years_decoded.csv', low_memory=False)

# Identify columns where decoding went wrong
# A column was wrongly decoded if it has MORE NaN in decoded than numeric
wrong_cols = []
for col in df_numeric.columns:
    if col in df_decoded.columns:
        numeric_nan = df_numeric[col].isna().sum()
        decoded_nan = df_decoded[col].isna().sum()
        if decoded_nan > numeric_nan + 100:  # allow small tolerance
            wrong_cols.append({
                'column': col,
                'numeric_nan': numeric_nan,
                'decoded_nan': decoded_nan,
                'extra_nan': decoded_nan - numeric_nan
            })

wrong_df = pd.DataFrame(wrong_cols).sort_values('extra_nan', ascending=False)
print(f"Columns with excess NaN after decoding: {len(wrong_df)}")
print(wrong_df.head(20).to_string(index=False))

Columns with excess NaN after decoding: 249
                     column  numeric_nan  decoded_nan  extra_nan
                     serial            0       135497     135497
   DURATION_RACKETSPORT_C06            0       135497     135497
      FREQUENCY_LEISURE_C08            0       135497     135497
      DAYS10P60_LEISURE_C08            0       135497     135497
        DAYS10P_LEISURE_C08            0       135497     135497
    MEMS7_ADVWATERSPORT_C07            0       135497     135497
 DURATION_ADVWATERSPORT_C07            0       135497     135497
FREQUENCY_ADVWATERSPORT_C07            0       135497     135497
  DAYS10P_ADVWATERSPORT_C07            0       135497     135497
      MEMS7_RACKETSPORT_C06            0       135497     135497
  FREQUENCY_RACKETSPORT_C06            0       135497     135497
          MEMS7_LEISURE_C08            0       135497     135497
  DAYS10P60_RACKETSPORT_C06            0       135497     135497
    DAYS10P_RACKETSPORT_C06            0      

In [22]:
# The correct approach — only decode genuinely categorical columns
# A column is categorical if its non-null numeric values are all integers
# and all those integers appear in the value labels

def is_categorical(col, df_numeric, value_labels):
    if col not in value_labels:
        return False
    labels = value_labels[col]
    # Get positive label keys (exclude sentinels)
    valid_label_keys = set(k for k in labels.keys() if k >= 0)
    if len(valid_label_keys) == 0:
        return False
    # Get actual non-null values in data
    actual_vals = set(df_numeric[col].dropna().unique())
    # Column is categorical if actual values are subset of label keys
    # (allowing for some values to be sentinel-converted to NaN already)
    overlap = actual_vals & valid_label_keys
    return len(overlap) > 0 and len(actual_vals) <= len(valid_label_keys) + 5

# Test on known cases
test_cols = ['Age9', 'Gend3', 'wt_final', 'MEMS7GR_ALL', 
             'DURATION_WALKALL_C01', 'VolAny', 'NSSEC5']
print("Categorical check on test columns:")
for col in test_cols:
    if col in df_numeric.columns:
        # Use Y8 value labels as reference
        _, meta_y8 = pyreadstat.read_sav(
            r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav",
            metadataonly=True
        )
        result = is_categorical(col, df_numeric, meta_y8.variable_value_labels)
        print(f"  {col:40} → {'CATEGORICAL' if result else 'CONTINUOUS'}")

Categorical check on test columns:
  Age9                                     → CATEGORICAL
  Gend3                                    → CATEGORICAL
  wt_final                                 → CONTINUOUS
  MEMS7GR_ALL                              → CATEGORICAL
  DURATION_WALKALL_C01                     → CONTINUOUS
  VolAny                                   → CATEGORICAL
  NSSEC5                                   → CATEGORICAL


In [23]:
import pyreadstat
import pandas as pd

# ── Reload numeric master ─────────────────────────────────────────────────────
df_numeric = pd.read_csv(r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_all_years.csv", low_memory=False)

# ── Load metadata for all 8 years ─────────────────────────────────────────────
year_savs = {
    '2015-16': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav",
    '2016-17': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8391-spss 2016-2017\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav",
    '2017-18': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8651-spss 2017-2018\spss\spss28\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav",
    '2018-19': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8652-spss 2018-2019\spss\spss28\active_lives_survey_nov_18-19_data_year_4_shared_20250103.sav",
    '2019-20': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8899-spss 2019-2020\spss\spss28\active_lives_survey_nov_19-20_data_year_5_shared_20250103.sav",
    '2020-21': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8993-spss 2020-2021\spss\spss28\active_lives_survey_nov_20-21_data_year_6_shared_20250103.sav",
    '2021-22': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9136-spss 2021-2022\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav",
    '2022-23': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav",
}

print("Loading metadata for all years...")
year_meta = {}
for year, path in year_savs.items():
    _, meta = pyreadstat.read_sav(path, metadataonly=True)
    year_meta[year] = meta.variable_value_labels
    print(f"  {year} ✓")

# ── Categorical detection function ────────────────────────────────────────────
def is_categorical(col, df_numeric, value_labels):
    if col not in value_labels:
        return False
    labels = value_labels[col]
    valid_label_keys = set(k for k in labels.keys() if k >= 0)
    if len(valid_label_keys) == 0:
        return False
    actual_vals = set(df_numeric[col].dropna().unique())
    overlap = actual_vals & valid_label_keys
    return len(overlap) > 0 and len(actual_vals) <= len(valid_label_keys) + 5

# ── Build unified value labels — use the year that has the most labels per col ─
print("\nBuilding unified value labels...")
unified_labels = {}
for col in df_numeric.columns:
    best_labels = {}
    for year, vl in year_meta.items():
        if col in vl:
            pos_labels = {k: v for k, v in vl[col].items() if k >= 0}
            if len(pos_labels) > len(best_labels):
                best_labels = pos_labels
    if best_labels:
        unified_labels[col] = best_labels

print(f"Variables with value labels: {len(unified_labels)}")

# ── Identify which columns to decode ──────────────────────────────────────────
cols_to_decode = [
    col for col in df_numeric.columns
    if is_categorical(col, df_numeric, unified_labels)
]
cols_to_skip = [
    col for col in df_numeric.columns
    if col not in cols_to_decode
]

print(f"Columns to decode (categorical): {len(cols_to_decode)}")
print(f"Columns to keep numeric:         {len(cols_to_skip)}")

# ── Build decoded dataset ──────────────────────────────────────────────────────
print("\nDecoding categorical columns...")
df_decoded = df_numeric.copy()

for col in cols_to_decode:
    df_decoded[col] = df_numeric[col].map(unified_labels[col])

print("✓ Decoding complete")

# ── Apply all EDA fixes ────────────────────────────────────────────────────────
# Fix 1: NSSEC5
df_decoded['NSSEC5'] = df_decoded['NSSEC5'].replace(
    'NS SEC 9: Students and other',
    'NS SEC 9: Students and other / unclassified'
)

# Fix 2: MEMS7GR_ALL label standardisation
df_decoded['MEMS7GR_ALL'] = df_decoded['MEMS7GR_ALL'].replace(
    'Insufficiently Active', 'Fairly Active'
)

# Fix 3: VolAny string codes
df_decoded['VolAny'] = df_decoded['VolAny'].replace({'0.0': 'No', '1.0': 'Yes'})

# Fix 4: Borough names (already in numeric dataset)
# borough column carries over from numeric ✓

print("\n✓ All EDA fixes applied")

# ── Verify ────────────────────────────────────────────────────────────────────
print("\n=== VERIFICATION ===")
print(f"Shape:           {df_decoded.shape}")
print(f"wt_final NaN%:   {df_decoded['wt_final'].isna().mean()*100:.1f}%")
print(f"Age9 sample:     {df_decoded['Age9'].value_counts().head(3).to_dict()}")
print(f"MEMS7GR_ALL:     {df_decoded['MEMS7GR_ALL'].value_counts().to_dict()}")
print(f"NSSEC5 unique:   {sorted(df_decoded['NSSEC5'].dropna().unique())}")
print(f"VolAny unique:   {df_decoded['VolAny'].value_counts().to_dict()}")

# Check wt_final has real values
print(f"\nwt_final stats:  min={df_decoded['wt_final'].min():.4f}, "
      f"max={df_decoded['wt_final'].max():.4f}, "
      f"mean={df_decoded['wt_final'].mean():.4f}")

# ── Save ──────────────────────────────────────────────────────────────────────
df_decoded.to_csv('active_lives_london_all_years_decoded.csv', index=False)
print("\n✓ Saved: active_lives_london_all_years_decoded.csv")

Loading metadata for all years...
  2015-16 ✓
  2016-17 ✓
  2017-18 ✓
  2018-19 ✓
  2019-20 ✓
  2020-21 ✓
  2021-22 ✓
  2022-23 ✓

Building unified value labels...
Variables with value labels: 366
Columns to decode (categorical): 366
Columns to keep numeric:         244

Decoding categorical columns...
✓ Decoding complete

✓ All EDA fixes applied

=== VERIFICATION ===
Shape:           (135497, 610)
wt_final NaN%:   0.0%
Age9 sample:     {'35-44': 28039, '25-34': 25177, '45-54': 23066}
MEMS7GR_ALL:     {'Active': 89628, 'Inactive': 30340, 'Fairly Active': 15529}
NSSEC5 unique:   ['NS SEC 1-2: Higher social groups', 'NS SEC 3-5: Middle social groups', 'NS SEC 6-8: Lower social groups', 'NS SEC 9: Students and other / unclassified']
VolAny unique:   {'No': 70502, 'Yes': 14786}

wt_final stats:  min=0.0706, max=37.0091, mean=1.6761

✓ Saved: active_lives_london_all_years_decoded.csv


In [24]:
# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 25)
pd.set_option('display.width', None)

# Load and display
df_all_decoded = pd.read_csv('active_lives_london_all_years_decoded.csv', low_memory=False, nrows=25)
print(f"Columns: {df_all_decoded.shape[1]}")
df_all_decoded.head(25)

Columns: 610


,serial,mode,group,Quarter,MonthTwelve,wt_final,xStrata,LA_code,LA_2023,LA_Old,Age5_2,Age9,Disab2_POP,Disab3,disty1_POP,disty10_POP,disty2_POP,disty3_POP,disty4_POP,disty5_POP,disty6_POP,disty7_POP,disty8_POP,disty9_POP,Educ6,Eth2,Eth7,EthAge4,Gend3,GendAge9,NSSEC5,NSSEC8,BMIG,MEMS7_ALL,MEMS7GR_ALL,MEMS7_SPORTCOUNT_A01,MEMS7GR_SPORTCOUNT_A01,MEMS7GR7_SPORTCOUNT_A01,MONTHS_12_WALKLEISURE_B01,DAYS10P_WALKLEISURE_B01,DAYS10P60_WALKLEISURE_B01,DAYS10P60GR_WALKLEISURE_B01,MEMS7_CAPPED_WALKLEISURE_B01,ACTYRA_WALKLEISURE_B01,ACTYRB_WALKLEISURE_B01,ACTYRC_WALKLEISURE_B01,FREQUENCY_WALKLEISURE_B01,DURATION_WALKLEISURE_B01,MEMS7_WALKLEISURE_B01,FREQUENCYGR_WALKLEISURE_B01,DURATIONGR_WALKLEISURE_B01,DURATION1PL_WALKLEISURE_B01,MEMS7GR_WALKLEISURE_B01,DAYS10PGR2_WALKLEISURE_B01,MONTHS_12_WALKTRAV_B02,DAYS10P_WALKTRAV_B02,DAYS10P60_WALKTRAV_B02,DAYS10P60GR_WALKTRAV_B02,MEMS7_CAPPED_WALKTRAV_B02,ACTYRA_WALKTRAV_B02,ACTYRB_WALKTRAV_B02,ACTYRC_WALKTRAV_B02,FREQUENCY_WALKTRAV_B02,DURATION_WALKTRAV_B02,MEMS7_WALKTRAV_B02,FREQUENCYGR_WALKTRAV_B02,DURATIONGR_WALKTRAV_B02,DURATION1PL_WALKTRAV_B02,MEMS7GR_WALKTRAV_B02,DAYS10PGR2_WALKTRAV_B02,MONTHS_12_CYCTRAV_B04,DAYS10P_CYCTRAV_B04,DAYS10P60_CYCTRAV_B04,DAYS10P60GR_CYCTRAV_B04,MEMS7_CAPPED_CYCTRAV_B04,ACTYRA_CYCTRAV_B04,ACTYRB_CYCTRAV_B04,ACTYRC_CYCTRAV_B04,FREQUENCY_CYCTRAV_B04,DURATION_CYCTRAV_B04,MEMS7_CYCTRAV_B04,FREQUENCYGR_CYCTRAV_B04,DURATIONGR_CYCTRAV_B04,DURATION1PL_CYCTRAV_B04,MEMS7GR_CYCTRAV_B04,DAYS10PGR2_CYCTRAV_B04,MONTHS_12_DANCEART_B05,DAYS10P_DANCEART_B05,DAYS10P60_DANCEART_B05,DAYS10P60GR_DANCEART_B05,ACTYRA_DANCEART_B05,ACTYRB_DANCEART_B05,ACTYRC_DANCEART_B05,FREQUENCY_DANCEART_B05,DURATION_DANCEART_B05,MEMS7_DANCEART_B05,FREQUENCYGR_DANCEART_B05,DURATIONGR_DANCEART_B05,DURATION1PL_DANCEART_B05,MEMS7GR_DANCEART_B05,DAYS10PGR2_DANCEART_B05,MONTHS_12_SPORTTRAD_B07,DAYS10P_SPORTTRAD_B07,DAYS10P60_SPORTTRAD_B07,DAYS10P60GR_SPORTTRAD_B07,ACTYRA_SPORTTRAD_B07,ACTYRB_SPORTTRAD_B07,ACTYRC_SPORTTRAD_B07,FREQUENCY_SPORTTRAD_B07,DURATION_SPORTTRAD_B07,MEMS7_SPORTTRAD_B07,FREQUENCYGR_SPORTTRAD_B07,DURATIONGR_SPORTTRAD_B07,DURATION1PL_SPORTTRAD_B07,MEMS7GR_SPORTTRAD_B07,MONTHS_12_WALKALL_C01,DAYS10P_WALKALL_C01,DAYS10P60_WALKALL_C01,DAYS10P60GR_WALKALL_C01,MEMS7_CAPPED_WALKALL_C01,ACTYRA_WALKALL_C01,ACTYRB_WALKALL_C01,ACTYRC_WALKALL_C01,FREQUENCY_WALKALL_C01,DURATION_WALKALL_C01,MEMS7_WALKALL_C01,FREQUENCYGR_WALKALL_C01,DURATIONGR_WALKALL_C01,DURATION1PL_WALKALL_C01,MEMS7GR_WALKALL_C01,MONTHS_12_ACTTRAV_C03,DAYS10P_ACTTRAV_C03,DAYS10P60_ACTTRAV_C03,DAYS10P60GR_ACTTRAV_C03,MEMS7_CAPPED_ACTTRAV_C03,ACTYRA_ACTTRAV_C03,ACTYRB_ACTTRAV_C03,ACTYRC_ACTTRAV_C03,FREQUENCY_ACTTRAV_C03,DURATION_ACTTRAV_C03,MEMS7_ACTTRAV_C03,FREQUENCYGR_ACTTRAV_C03,DURATIONGR_ACTTRAV_C03,DURATION1PL_ACTTRAV_C03,MEMS7GR_ACTTRAV_C03,MONTHS_12_DANCEALL_C04,DAYS10P_DANCEALL_C04,DAYS10P60_DANCEALL_C04,DAYS10P60GR_DANCEALL_C04,ACTYRA_DANCEALL_C04,ACTYRB_DANCEALL_C04,ACTYRC_DANCEALL_C04,FREQUENCY_DANCEALL_C04,DURATION_DANCEALL_C04,MEMS7_DANCEALL_C04,FREQUENCYGR_DANCEALL_C04,DURATIONGR_DANCEALL_C04,DURATION1PL_DANCEALL_C04,MEMS7GR_DANCEALL_C04,MONTHS_12_TEAMSPORT_C05,DAYS10P_TEAMSPORT_C05,DAYS10P60_TEAMSPORT_C05,DAYS10P60GR_TEAMSPORT_C05,ACTYRA_TEAMSPORT_C05,ACTYRB_TEAMSPORT_C05,ACTYRC_TEAMSPORT_C05,FREQUENCY_TEAMSPORT_C05,DURATION_TEAMSPORT_C05,MEMS7_TEAMSPORT_C05,FREQUENCYGR_TEAMSPORT_C05,DURATIONGR_TEAMSPORT_C05,DURATION1PL_TEAMSPORT_C05,MEMS7GR_TEAMSPORT_C05,MINS_SESS_TEAMSPORT_C05,MONTHS_12_RACKETSPORT_C06,DAYS10P_RACKETSPORT_C06,DAYS10P60_RACKETSPORT_C06,DAYS10P60GR_RACKETSPORT_C06,ACTYRA_RACKETSPORT_C06,ACTYRB_RACKETSPORT_C06,ACTYRC_RACKETSPORT_C06,FREQUENCY_RACKETSPORT_C06,DURATION_RACKETSPORT_C06,MEMS7_RACKETSPORT_C06,FREQUENCYGR_RACKETSPORT_C06,DURATIONGR_RACKETSPORT_C06,DURATION1PL_RACKETSPORT_C06,MEMS7GR_RACKETSPORT_C06,MONTHS_12_ADVWATERSPORT_C07,DAYS10P_ADVWATERSPORT_C07,DAYS10P60_ADVWATERSPORT_C07,DAYS10P60GR_ADVWATERSPORT_C07,ACTYRA_ADVWATERSPORT_C07,ACTYRB_ADVWATERSPORT_C07,ACTYRC_ADVWATERSPORT_C07,FREQUENCY_ADV

In [28]:
df_numeric = pd.read_csv(r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_all_years.csv", low_memory=False)

print("Sample serial numbers by year:")
for year in sorted(df_numeric['survey_year'].unique()):
    sample = df_numeric[df_numeric['survey_year'] == year]['serial'].head(3).tolist()
    print(f"  {year}: {sample}")

print(f"\nOverall serial range:")
print(f"  Min: {df_numeric['serial'].min():.0f}")
print(f"  Max: {df_numeric['serial'].max():.0f}")



Sample serial numbers by year:
  2015-16: [151190024990481.0, 151190024991282.0, 151190024994382.0]
  2016-17: [160290116991681.0, 160490128184481.0, 160590037872981.0]
  2017-18: [170390237749481.0, 170390287888181.0, 170390287888182.0]
  2018-19: [180390156596381.0, 180390156596382.0, 180590024971502.0]
  2019-20: [180990336530661.0, 181090055871662.0, 181290286875202.0]
  2020-21: [191190306702332.0, 191290196496291.0, 200190286661921.0]
  2021-22: [201190335972081.0, 201290055318612.0, 210190195544061.0]
  2022-23: [220290145933162.0, 220490175876851.0, 220590035585412.0]

Overall serial range:
  Min: 151190024990481
  Max: 231090335927772
survey_year
2015-16    25
Name: count, dtype: int64


In [29]:
df_all_decoded = pd.read_csv(r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_all_years_decoded.csv", low_memory=False)
print(df_all_decoded['survey_year'].value_counts().sort_index())

survey_year
2015-16    19620
2016-17    19248
2017-18    15967
2018-19    15889
2019-20    16091
2020-21    16028
2021-22    16139
2022-23    16515
Name: count, dtype: int64


In [31]:
print(f"'borough' column: {df_all_decoded['borough'].nunique()} unique values")
print(df_all_decoded['borough'].value_counts().head(5))

print(f"\n'LA_code' column: {df_all_decoded['LA_code'].nunique()} unique values")
print(df_all_decoded['LA_code'].value_counts().head(5))

'borough' column: 32 unique values
borough
Greenwich               5472
Lambeth                 5025
Haringey                5025
Enfield                 4983
Barking and Dagenham    4968
Name: count, dtype: int64

'LA_code' column: 78 unique values
LA_code
9.0      4953
44.0     4020
139.0    4010
286.0    3524
8.0      3514
Name: count, dtype: int64


In [2]:
import pyreadstat
import pandas as pd

SAV_Y1 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"

# Check 1: Does MONTHS_12_CYCALL_C02 exist in Y1 metadata at all?
_, meta_y1 = pyreadstat.read_sav(SAV_Y1, metadataonly=True)

exists_in_meta = 'MONTHS_12_CYCALL_C02' in meta_y1.column_names
print(f"MONTHS_12_CYCALL_C02 exists in Y1 metadata: {exists_in_meta}")

if exists_in_meta:
    label = meta_y1.column_names_to_labels.get('MONTHS_12_CYCALL_C02', 'no label')
    print(f"Label: {label}")
    
    # Check 2: Load just this column from Y1 SPSS directly
    df_test, _ = pyreadstat.read_sav(
        SAV_Y1,
        usecols=['MONTHS_12_CYCALL_C02', 'MONTHS_12_CYCTRAV_B04', 
                 'MONTHS_12_CYCLEISURE_N01'],
        apply_value_formats=False
    )
    print(f"\nDirect from Y1 SPSS file:")
    print(f"MONTHS_12_CYCALL_C02  — NaN: {df_test['MONTHS_12_CYCALL_C02'].isna().sum()} / {len(df_test)}")
    print(f"MONTHS_12_CYCTRAV_B04 — NaN: {df_test['MONTHS_12_CYCTRAV_B04'].isna().sum()} / {len(df_test)}")
    print(f"\nMONTHS_12_CYCALL_C02 value counts:")
    print(df_test['MONTHS_12_CYCALL_C02'].value_counts(dropna=False))
    
else:
    print("Variable does NOT exist in Y1 — confirmed genuinely absent")
    print("\nY1 cycling variables available:")
    cyc = [v for v in meta_y1.column_names if 'CYCALL' in v.upper()]
    print(cyc)

MONTHS_12_CYCALL_C02 exists in Y1 metadata: False
Variable does NOT exist in Y1 — confirmed genuinely absent

Y1 cycling variables available:
['DURATION_CYCALL_NOEXBIKE_D05', 'FREQUENCYGR_CYCALL_NOEXBIKE_D05', 'DURATIONGR_CYCALL_NOEXBIKE_D05']


In [ ]:
X